In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("alx-spark").getOrCreate()
spark

In [3]:
raw_df = spark.read.csv("data/bitstamp(USD)20211-2024.csv", header=True)
raw_df.printSchema()
raw_df.show(5)

root
 |-- Timestamp: string (nullable = true)
 |-- Open: string (nullable = true)
 |-- High: string (nullable = true)
 |-- Low: string (nullable = true)
 |-- Close: string (nullable = true)
 |-- Volume (BTC): string (nullable = true)
 |-- Volume (Currency): string (nullable = true)
 |-- Weighted Price: string (nullable = true)

+------------------+----+----+----+-----+------------+-----------------+--------------+
|         Timestamp|Open|High| Low|Close|Volume (BTC)|Volume (Currency)|Weighted Price|
+------------------+----+----+----+-----+------------+-----------------+--------------+
|2011-09-13 0:00:00| 5.8|   6|5.65| 5.97|       58.37|            346.1|          5.93|
|2011-09-14 0:00:00|5.58|5.72|5.52| 5.53|       61.15|           341.85|          5.59|
|2011-09-15 0:00:00|5.12|5.24|   5| 5.13|       80.14|           408.26|          5.09|
|2011-09-16 0:00:00|4.82|4.87| 4.8| 4.85|       39.91|           193.76|          4.85|
|2011-09-17 0:00:00|4.87|4.87|4.87| 4.87|         0.3|

In [4]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

btc_schema = StructType([
    StructField("timestamp", StringType()),   # read as text first, convert below
    StructField("open", DoubleType()),
    StructField("high", DoubleType()),
    StructField("low", DoubleType()),
    StructField("close", DoubleType()),
    StructField("volume_btc", DoubleType()),
    StructField("volume_currency", DoubleType()),
    StructField("weighted_price", DoubleType())
])

btc = spark.read.csv("data/bitstamp(USD)20211-2024.csv", header=True, schema=btc_schema)

btc = btc.withColumn("timestamp", F.to_timestamp(F.col("timestamp"), "yyyy-MM-dd H:mm:ss"))
btc = btc.withColumn("year", F.year("timestamp"))

btc.write.mode("overwrite").parquet("data/bitcoin_parquet")

In [5]:
df = spark.read.parquet("data/bitcoin_parquet")
df.printSchema()
df.show(5)

root
 |-- timestamp: timestamp (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume_btc: double (nullable = true)
 |-- volume_currency: double (nullable = true)
 |-- weighted_price: double (nullable = true)
 |-- year: integer (nullable = true)

+-------------------+----+----+----+-----+----------+---------------+--------------+----+
|          timestamp|open|high| low|close|volume_btc|volume_currency|weighted_price|year|
+-------------------+----+----+----+-----+----------+---------------+--------------+----+
|2011-09-13 00:00:00| 5.8| 6.0|5.65| 5.97|     58.37|          346.1|          5.93|2011|
|2011-09-14 00:00:00|5.58|5.72|5.52| 5.53|     61.15|         341.85|          5.59|2011|
|2011-09-15 00:00:00|5.12|5.24| 5.0| 5.13|     80.14|         408.26|          5.09|2011|
|2011-09-16 00:00:00|4.82|4.87| 4.8| 4.85|     39.91|         193.76|          4.85|2011|
|201

In [6]:
practice = btc.select("timestamp", "open", "high", "low", "close", "volume_btc", "volume_currency", "weighted_price", "year")

practice = practice.withColumnRenamed("volume_currency", "volume_usd")

practice = practice.withColumn("spread", F.col("high") - F.col("low"))

practice = practice.withColumn("exchange", F.lit("Bitstamp"))

practice = practice.drop("weighted_price")

practice.printSchema()
practice.show(5)

root
 |-- timestamp: timestamp (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume_btc: double (nullable = true)
 |-- volume_usd: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- spread: double (nullable = true)
 |-- exchange: string (nullable = false)

+-------------------+----+----+----+-----+----------+----------+----+-------------------+--------+
|          timestamp|open|high| low|close|volume_btc|volume_usd|year|             spread|exchange|
+-------------------+----+----+----+-----+----------+----------+----+-------------------+--------+
|2011-09-13 00:00:00| 5.8| 6.0|5.65| 5.97|     58.37|     346.1|2011|0.34999999999999964|Bitstamp|
|2011-09-14 00:00:00|5.58|5.72|5.52| 5.53|     61.15|    341.85|2011|0.20000000000000018|Bitstamp|
|2011-09-15 00:00:00|5.12|5.24| 5.0| 5.13|     80.14|    408.26|2011| 0.2400000000000002|Bitstamp|
|2011-09-16 

In [7]:
from datetime import date
filter_date = date(2020, 1, 1)

In [8]:
practice.select('timestamp', 'close', 'spread').where(F.col('timestamp') > filter_date).show(5, True)

+-------------------+-------+------------------+
|          timestamp|  close|            spread|
+-------------------+-------+------------------+
|2020-01-02 00:00:00|6950.56| 284.9399999999996|
|2020-01-03 00:00:00|7338.91| 548.7800000000007|
|2020-01-04 00:00:00|7344.48|140.07000000000062|
|2020-01-05 00:00:00| 7356.7|             185.0|
|2020-01-06 00:00:00|7762.74|474.53999999999996|
+-------------------+-------+------------------+
only showing top 5 rows


In [10]:
# How many days had close > 10000 and spread > 1000? Use .count()
# .count() is an action that returns a number
filter_1 = F.col("close") > 10000
filter_2 = F.col("spread") > 1000
practice.select('timestamp', 'close', 'spread').where(filter_1 & filter_2).count()

953

In [16]:
"""practice.dropna().groupBy(F.year('timestamp'))\
    .agg({'high': 'max', 'volume_usd': 'sum', 'close': 'mean'})\
    .select(F.col('year(timestamp)').alias('year_agg'), 
            F.col('max(high)').alias('max_high'), 
            F.col('sum(volume_usd)').alias('total_volume')\
    .orderBy('year(timestamp)').show(10, False)"""

practice.dropna() \
    .groupBy("year") \
    .agg(F.count("*").alias("days"),
         F.max("high").alias("max_high"),
         F.avg("close").alias("avg_close"),
         F.sum("volume_usd").alias("total_volume")) \
    .orderBy("year") \
    .show(20, False)

+----+----+--------+------------------+---------------------+
|year|days|max_high|avg_close         |total_volume         |
+----+----+--------+------------------+---------------------+
|2011|92  |15.0    |3.829347826086958 |28108.93000000001    |
|2012|366 |16.41   |8.254426229508196 |5764800.290000003    |
|2013|365 |1163.0  |188.73597260273982|1.551177487E9        |
|2014|365 |995.0   |525.0136712328763 |2.6152928093400025E9 |
|2015|362 |502.0   |272.2394475138122 |1.5454445301100001E9 |
|2016|366 |980.74  |566.7835792349732 |1.112463797099999E9  |
|2017|365 |19666.0 |3972.172547945205 |2.1775822446599983E10|
|2018|365 |17234.99|7522.2250410958895|3.0746672206330006E10|
|2019|365 |13880.0 |7363.344054794521 |2.315853609881001E10 |
|2020|366 |29300.0 |11115.837049180329|3.3290575549870014E10|
|2021|365 |69000.0 |47431.34630136984 |7.964646106038994E10 |
|2022|365 |48234.0 |28188.338602739725|2.2665352439169983E10|
|2023|365 |44729.0 |28858.728767123288|1.9396700940080013E10|
|2024|17